# Объединение данных из разных источников

В этом ноутбуке объединяются данные о товарах «Смешарики», собранные с маркетплейсов и дополнительных сайтов.

Основная задача - привести таблицы из разных источников к единой структуре и сохранить общий файл для дальнейшего анализа.

Источники данных:

- основной датасет с маркетплейсов Wildberries и Ozon;
- сайт «Два мяча»;
- сайт Smlerch;
- данные Ювелирного дома SASONKO / Riki Collection.

## 1. Импорт библиотек

Подключаются библиотеки для работы с таблицами, числовыми значениями, регулярными выражениями и загрузкой файлов в Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files

## 2. Загрузка исходных файлов

В Colab загружаются Excel-файлы, которые были получены на предыдущем этапе парсинга. Далее задаются названия файлов, с которыми будет работать ноутбук.

In [ ]:
uploaded = files.upload()

base_file = 'smeshariki_marketplaces_final (2).xlsx'
dvamyacha_file = 'smeshariki_final (2).xlsx'
smlerch_file = 'smlerch_full_data (1).xlsx'
sasonko_file = 'riki_selenium_fixed (1).xlsx' 

## 3. Чтение таблиц

Каждый файл считывается в отдельный DataFrame. После загрузки выводятся размеры таблиц, чтобы проверить, что данные были считаны корректно.

In [ ]:
base = pd.read_excel(base_file)
dvamyacha = pd.read_excel(dvamyacha_file)
smlerch = pd.read_excel(smlerch_file)
sasonko = pd.read_excel(sasonko_file)

print('Размеры исходных таблиц:')
print('Основа:', base.shape)
print('Два мяча:', dvamyacha.shape)
print('Smlerch:', smlerch.shape)
print('SASONKO:', sasonko.shape)

print('\nКолонки основной таблицы:')
print(base.columns.tolist())

## 4. Вспомогательные функции

Создаются функции для подготовки данных:

- `clean_price()` приводит цену к числовому формату;
- `make_empty_like_base()` создаёт пустую таблицу со структурой основного датасета;
- `fill_if_column_exists()` безопасно заполняет столбцы, если они есть в основной таблице.

In [ ]:
def clean_price(value):
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace('\xa0', '')
    value = value.replace(' ', '')
    value = value.replace('₽', '')
    value = value.replace('руб.', '')
    value = value.replace('руб', '')
    value = value.replace(',', '.')

    numbers = re.findall(r'\d+\.?\d*', value)

    if len(numbers) == 0:
        return np.nan

    return float(numbers[0])


def make_empty_like_base(base_df, n_rows):
    return pd.DataFrame(columns=base_df.columns, index=range(n_rows))


def fill_if_column_exists(df, column_name, values):
    if column_name in df.columns:
        df[column_name] = values

## 5. Подготовка данных сайта «Два мяча»

Данные о кедах «Смешарики» приводятся к структуре основной таблицы. Для товаров создаются технические идентификаторы, заполняются название, цена, продавец, источник и ссылка.

In [ ]:
dvamyacha_new = make_empty_like_base(base, len(dvamyacha))

fill_if_column_exists(dvamyacha_new, 'product_id', ['dvamyacha_' + str(i + 1) for i in range(len(dvamyacha))])
fill_if_column_exists(dvamyacha_new, 'product_name', dvamyacha['Товар'])
fill_if_column_exists(dvamyacha_new, 'brand', 'Смешарики')
fill_if_column_exists(dvamyacha_new, 'source_name', 'Два мяча')
fill_if_column_exists(dvamyacha_new, 'source_type', 'additional_store')
fill_if_column_exists(dvamyacha_new, 'source_page', 'Два мяча')
fill_if_column_exists(dvamyacha_new, 'seller', 'Два мяча')
fill_if_column_exists(dvamyacha_new, 'price', dvamyacha['Цена'].apply(clean_price))
fill_if_column_exists(dvamyacha_new, 'discount_price', dvamyacha['Цена'].apply(clean_price))
fill_if_column_exists(dvamyacha_new, 'category_name', 'Кеды')
fill_if_column_exists(dvamyacha_new, 'marketplace', 'Два мяча')
fill_if_column_exists(dvamyacha_new, 'link', dvamyacha['Ссылка'])

display(dvamyacha_new.head())

## 6. Подготовка данных Smlerch

Данные с сайта Smlerch также приводятся к единому формату. Так как товары относятся к футболкам, категория заполняется значением `Футболки`.

In [ ]:
smlerch_new = make_empty_like_base(base, len(smlerch))

fill_if_column_exists(smlerch_new, 'product_id', ['smlerch_' + str(i + 1) for i in range(len(smlerch))])
fill_if_column_exists(smlerch_new, 'product_name', smlerch['Название'])
fill_if_column_exists(smlerch_new, 'brand', 'Смешарики')
fill_if_column_exists(smlerch_new, 'source_name', 'Smlerch')
fill_if_column_exists(smlerch_new, 'source_type', 'additional_store')
fill_if_column_exists(smlerch_new, 'source_page', 'Smlerch')
fill_if_column_exists(smlerch_new, 'seller', 'Smlerch')
fill_if_column_exists(smlerch_new, 'price', smlerch['Цена'].apply(clean_price))
fill_if_column_exists(smlerch_new, 'discount_price', smlerch['Цена'].apply(clean_price))
fill_if_column_exists(smlerch_new, 'category_name', 'Футболки')
fill_if_column_exists(smlerch_new, 'marketplace', 'Smlerch')
fill_if_column_exists(smlerch_new, 'link', smlerch['Ссылка'])

display(smlerch_new.head())

## 7. Подготовка данных Ювелирного дома SASONKO

Для данных SASONKO выполняется дополнительная очистка: удаляются служебные строки и дубликаты по ссылке на товар. Название товара формируется из категории и названия персонажа, например: `Запонки Крош`.

In [ ]:
sasonko = sasonko.copy()

sasonko = sasonko[
    sasonko['Название'].notna()
    & ~sasonko['Название'].astype(str).str.contains('#order', case=False, na=False)
]

sasonko = sasonko.drop_duplicates(subset=['Ссылка на товар'])

sasonko_new = make_empty_like_base(base, len(sasonko))

sasonko_product_name = (
    sasonko['Категория'].astype(str).str.strip().str.title()
    + ' '
    + sasonko['Название'].astype(str).str.strip()
)

fill_if_column_exists(sasonko_new, 'product_id', ['sasonko_' + str(i + 1) for i in range(len(sasonko))])
fill_if_column_exists(sasonko_new, 'product_name', sasonko_product_name)
fill_if_column_exists(sasonko_new, 'brand', 'Смешарики')
fill_if_column_exists(sasonko_new, 'source_name', 'Ювелирный дом SASONKO')
fill_if_column_exists(sasonko_new, 'source_type', 'additional_store')
fill_if_column_exists(sasonko_new, 'source_page', 'Riki Collection')
fill_if_column_exists(sasonko_new, 'seller', 'Ювелирный дом SASONKO')
fill_if_column_exists(sasonko_new, 'price', sasonko['Цена (руб)'].apply(clean_price))
fill_if_column_exists(sasonko_new, 'discount_price', sasonko['Цена (руб)'].apply(clean_price))
fill_if_column_exists(sasonko_new, 'category_name', sasonko['Категория'])
fill_if_column_exists(sasonko_new, 'marketplace', 'Ювелирный дом SASONKO')
fill_if_column_exists(sasonko_new, 'link', sasonko['Ссылка на товар'])

display(sasonko_new.head())

## 8. Объединение таблиц

Все подготовленные таблицы объединяются в один датасет. Так как структура колонок заранее приведена к основной таблице, объединение выполняется через `pd.concat()`.

In [ ]:
merged = pd.concat(
    [base, dvamyacha_new, smlerch_new, sasonko_new],
    ignore_index=True
)

print('Размер итоговой таблицы:', merged.shape)
display(merged.head())

## 9. Проверка результата

Проверяем, сколько товаров пришло из каждого источника, и выводим последние строки таблицы, где должны находиться добавленные товары из дополнительных магазинов.

In [ ]:
print('Количество товаров по marketplace:')

if 'marketplace' in merged.columns:
    display(merged['marketplace'].value_counts(dropna=False).to_frame('products_count'))

print('Последние добавленные строки:')
display(merged.tail(20))

## 10. Сохранение итогового файла

После объединения данных итоговый датасет сохраняется в Excel-файл. Этот файл можно использовать в основном ноутбуке анализа.

In [ ]:
output_file = 'smeshariki_all_sources_merged.xlsx'

merged.to_excel(output_file, index=False)

files.download(output_file)